# 08 — Florida Cedar Key mangrove transfer

**Hypothesis.** Fine-tuning on Florida black mangrove trees improves single-tree mask IoU for subtropical mangroves.

**Runtime.** CPU for ingest; A100 for train

**Config.** `configs/experiment/cedar_key_lora.yaml`

Train → frozen shared benchmark (`configs/experiment/compare_val.yaml`) → comparison table → optional Colab `runtime.unassign()`.

Benchmark is **crop IoU** (single-tree set), not Wytham tiles. For tropical plot instances in our PLY schema, prefer Robson Creek via notebook 06 + `multisite_tropical.yaml`.


## 0. Options


In [ ]:
# Colab: mount Drive / clone repo first (see notebooks/00_colab.ipynb), then open this notebook from the repo.
TRAIN_CONFIG = "configs/experiment/cedar_key_lora.yaml"
COMPARE_CONFIG = "configs/experiment/compare_val.yaml"
SKIP_TRAIN = False          # True to only re-benchmark an existing results/<run>/best.safetensors
AUTO_SHUTDOWN = True        # releases the Colab VM after results are written (set False while debugging)
CROP_ONLY_BENCHMARK = True  # True for Cedar Key (no Wytham-style tiles)


## 1. Repo root + VRAM probe


In [ ]:
import os, sys
from pathlib import Path
root = Path.cwd().resolve()
for _ in range(4):
    if (root / "src" / "treeseg").is_dir():
        break
    root = root.parent
os.chdir(root)
sys.path.insert(0, "src")

from treeseg.engine.experiment_run import ensure_repo_root, probe_resources, reset_cuda_peaks
import torch

print("cwd:", ensure_repo_root())
reset_cuda_peaks()
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
probe_resources("notebook_start")


## 1b. Data prepare (CPU-friendly; skip if already on Drive)


In [ ]:
# Place the Figshare LAS archive under data/raw/cedar_key_mangrove (doi:10.6084/m9.figshare.31324393), then:
from pathlib import Path
import subprocess, sys

def sh(cmd):
    print("+", cmd, flush=True)
    subprocess.check_call(cmd, shell=True)

if not Path("data/cedar_key_mangrove/crops/train").is_dir():
    sh(f"{sys.executable} scripts/ingest_cedar_key.py configs/data/cedar_key_mangrove.yaml")
print("Cedar Key crops ready")
print("Blueflux Everglades TLS is unlabeled — not used for supervised train "
      "(https://www.earthdata.nasa.gov/data/catalog/ornl-cloud-tls-lidar-blueflux-mangroves-2311-1)")


## 2. Train + shared benchmark + compare


In [ ]:
from treeseg.engine.experiment_run import run_train_and_compare
from IPython.display import display

table = run_train_and_compare(
    TRAIN_CONFIG,
    COMPARE_CONFIG,
    skip_train=SKIP_TRAIN,
    auto_shutdown=AUTO_SHUTDOWN,
    crop_only_benchmark=CROP_ONLY_BENCHMARK,
)
display(table)


## Compute notes

After the first successful run, inspect `results/<run>/resource_probe.json` for peak GPU GB. If peak is well under 14 GB, later LoRA twins may use a cheaper GPU than A100. Always keep checkpoints on Drive before `AUTO_SHUTDOWN`.
